In [2]:
import pandas as pd
import numpy as np
import pickle
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# load movies
movies = pd.read_csv('../data/processed/movies_final.csv')
movies['genres'] = movies['genres'].apply(ast.literal_eval)
movies['keywords'] = movies['keywords'].apply(ast.literal_eval)
movies['top_cast'] = movies['top_cast'].apply(ast.literal_eval)

# load collaborative filtering artifacts
with open('../models/svd_predictions.pkl', 'rb') as f:
    predicted_ratings = pickle.load(f)

with open('../models/user_id_to_idx.pkl', 'rb') as f:
    user_id_to_idx = pickle.load(f)

with open('../models/idx_to_movie_id.pkl', 'rb') as f:
    idx_to_movie_id = pickle.load(f)

with open('../models/movie_encoder.pkl', 'rb') as f:
    movie_encoder = pickle.load(f)

# load links
links = pd.read_csv('../data/processed/links_cleaned.csv')
ratings_sample = pd.read_csv('../data/processed/ratings_sample.csv')

print("Everything loaded!")
print("Movies:", movies.shape)
print("Predicted ratings:", predicted_ratings.shape)
print("Users in collab model:", len(user_id_to_idx))

Everything loaded!
Movies: (4800, 13)
Predicted ratings: (20000, 5000)
Users in collab model: 20000


In [4]:
# rebuild soup and cosine similarity
# (we could load from pkl but rebuilding is cleaner and faster than you think)

def clean_name(name):
    return name.replace(" ", "").lower()

def build_soup(row):
    genres = [clean_name(g) for g in row['genres']]
    keywords = [clean_name(k) for k in row['keywords']]
    cast = [clean_name(c) for c in row['top_cast']]
    director = clean_name(row['director'])
    overview = row['overview'].lower()
    return ' '.join(genres) + ' ' + \
           ' '.join(keywords) + ' ' + \
           ' '.join(cast) + ' ' + \
           director + ' ' + director + ' ' + \
           overview

movies['soup'] = movies.apply(build_soup, axis=1)

tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(movies['soup'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

print("Content model rebuilt!")
print("Cosine sim matrix:", cosine_sim.shape)

Content model rebuilt!
Cosine sim matrix: (4800, 4800)


In [8]:
def get_hybrid_recommendations(user_id, title, n=10):
    """
    Combines content-based and collaborative filtering.
    
    user_id: MovieLens user ID
    title: a movie the user likes (seed movie for content signal)
    n: number of recommendations to return
    """
    
    # ── CONTENT SIGNAL ──────────────────────────────────────────
    if title not in indices:
        return f"Movie '{title}' not found"
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:51]  # top 50 similar movies as candidates
    
    candidate_indices = [i[0] for i in sim_scores]
    candidate_scores = {i[0]: i[1] for i in sim_scores}
    
    # build candidate dataframe
    candidates = movies.iloc[candidate_indices][['title', 'id', 'genres', 
                                                   'vote_average', 'weighted_score']].copy()
    candidates['content_score'] = candidates.index.map(candidate_scores)
    
    # ── COLLABORATIVE SIGNAL ─────────────────────────────────────
    user_known = user_id in user_id_to_idx
    
    if user_known:
        user_idx = user_id_to_idx[user_id]
        
        # get already rated movies to exclude
        already_rated = set(ratings_sample[
            ratings_sample['userId'] == user_id]['movieId'].values)
        
        # for each candidate, get collaborative predicted rating
        collab_scores = []
        for _, row in candidates.iterrows():
            tmdb_id = row['id']
            link_row = links[links['tmdbId'] == tmdb_id]
            
            if len(link_row) > 0:
                movie_id = link_row['movieId'].values[0]
                if movie_id in already_rated:
                    collab_scores.append(np.nan)
                elif movie_id in movie_encoder.classes_:
                    movie_idx = np.where(movie_encoder.classes_ == movie_id)[0][0]
                    collab_scores.append(predicted_ratings[user_idx][movie_idx])
                else:
                    collab_scores.append(np.nan)
            else:
                collab_scores.append(np.nan)
        
        candidates['collab_score'] = collab_scores
        
        # dynamic alpha based on how much we know about this user
        user_rating_count = len(ratings_sample[ratings_sample['userId'] == user_id])
        # more ratings = trust collaborative more
        alpha = max(0.3, 1 - (user_rating_count / 1000))
        
    else:
        # unknown user — pure content based
        candidates['collab_score'] = np.nan
        alpha = 1.0
    
    # ── COMBINE SCORES ───────────────────────────────────────────
    # normalize content score to 0-1
    cs_max = candidates['content_score'].max()
    cs_min = candidates['content_score'].min()
    candidates['content_norm'] = (candidates['content_score'] - cs_min) / (cs_max - cs_min)
    
    # normalize collab score to 0-1
    if user_known and candidates['collab_score'].notna().any():
        col_max = candidates['collab_score'].max()
        col_min = candidates['collab_score'].min()
        candidates['collab_norm'] = (candidates['collab_score'] - col_min) / (col_max - col_min)
    else:
        candidates['collab_norm'] = 0
    
    # weighted combination
    candidates['collab_norm'] = candidates['collab_norm'].fillna(0)
    candidates['final_score'] = (alpha * candidates['content_norm'] + 
                                  (1 - alpha) * candidates['collab_norm'])
    
    # sort and return top n
    result = candidates.sort_values('final_score', ascending=False).head(n)
    
    return result[['title', 'genres', 'vote_average', 
                   'content_score', 'collab_score', 
                   'final_score', ]].reset_index(drop=True)

In [10]:
# pick a real user from our sample
test_user = int(ratings_sample['userId'].iloc[100])

print(f"Testing hybrid for user {test_user} seeded with Interstellar")
print(f"Alpha will be dynamic based on their rating history\n")

result = get_hybrid_recommendations(test_user, "Interstellar", n=10)
print(result)

Testing hybrid for user 3 seeded with Interstellar
Alpha will be dynamic based on their rating history

                          title  \
0                Silent Running   
1                       Solaris   
2                     Apollo 13   
3          The Thirteenth Floor   
4           You Only Live Twice   
5               The Right Stuff   
6  A.I. Artificial Intelligence   
7                 Lost in Space   
8                       Lockout   
9                     Moonraker   

                                           genres  vote_average  \
0             [Adventure, Drama, Science Fiction]           6.3   
1    [Drama, Science Fiction, Adventure, Mystery]           7.7   
2                                         [Drama]           7.3   
3            [Thriller, Science Fiction, Mystery]           6.8   
4                   [Action, Thriller, Adventure]           6.5   
5                                [Drama, History]           7.3   
6             [Drama, Science Fiction, Ad

In [12]:
print(len(ratings_sample[ratings_sample['userId'] == 3]))

647


In [14]:
# pure content recommendations
def get_content_only(title, n=10):
    if title not in indices:
        return f"Movie '{title}' not found"
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]
    movie_indices = [i[0] for i in sim_scores]
    result = movies[['title', 'vote_average']].iloc[movie_indices].copy()
    result['similarity'] = [round(i[1], 3) for i in sim_scores]
    return result.reset_index(drop=True)

print("=== PURE CONTENT (same for every user) ===")
print(get_content_only("Interstellar"))
print()
print("=== HYBRID (personalized for user 3) ===")
print(get_hybrid_recommendations(3, "Interstellar")[['title', 'vote_average', 'final_score']])

=== PURE CONTENT (same for every user) ===
                          title  vote_average  similarity
0  Space Pirate Captain Harlock           6.5       0.161
1                       Contact           7.2       0.158
2                     Apollo 13           7.3       0.152
3                   The Martian           7.6       0.144
4                Silent Running           6.3       0.143
5                 Space Cowboys           6.3       0.137
6                     Inception           8.1       0.130
7                    Armageddon           6.4       0.130
8                      Insomnia           6.8       0.125
9                  The Prestige           8.0       0.125

=== HYBRID (personalized for user 3) ===
                          title  vote_average  final_score
0                Silent Running           6.3     0.736353
1                       Solaris           7.7     0.705209
2                     Apollo 13           7.3     0.692709
3          The Thirteenth Floor          

In [16]:
import pickle

# save cosine similarity matrix
with open('../models/cosine_sim.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)

# save indices mapping
with open('../models/indices.pkl', 'wb') as f:
    pickle.dump(indices, f)

# save final movies dataframe
movies.to_csv('../data/processed/movies_final.csv', index=False)

print("Hybrid model artifacts saved!")
print("Files in models/:")
import os
for f in os.listdir('../models/'):
    size = os.path.getsize(f'../models/{f}') / 1024 / 1024
    print(f"  {f}: {size:.1f} MB")

Hybrid model artifacts saved!
Files in models/:
  cosine_sim.pkl: 175.8 MB
  idx_to_movie_id.pkl: 0.1 MB
  indices.pkl: 0.1 MB
  movie_encoder.pkl: 0.0 MB
  svd_predictions.pkl: 762.9 MB
  user_encoder.pkl: 0.2 MB
  user_id_to_idx.pkl: 0.4 MB
